# Kinerja pipeline Redis: chunking, embedding, index, latensi query

Memakai `zul.evaluation` (lihat panduan *Mengevaluasi retrieval dan RAG* di dokumentasi Zul). Tidak ada kunci atau alamat server di sini; semuanya dari `.env` di akar repo. Jalankan kernel dari folder `framework/studies/retrieval_evaluation`.

In [ ]:
import json

import pandas as pd
from dotenv import load_dotenv

from zul.evaluation.config import RedisSettings
from zul.evaluation.documents import (
    chunk_per_page,
    chunk_recursive,
    chunks_to_records,
    find_pdfs,
    length_stats,
)
from zul.evaluation.indexing import embed_records, redis_schema
from zul.evaluation.latency import measure_latency, save_times_csv
from zul.evaluation.retrievers import RedisRetriever
from zul.utilities.embedding_service import AIService
from zul.utilities.script_helper.eval_performance import TimerDecorator

load_dotenv("../../../.env")
ai = AIService.from_env()
DIMS = ai.embed("uji").dimensions
print("dimensi embedding:", DIMS)

## 1. Chunking: per halaman vs rekursif

Lama tiap pemotongan dicatat `TimerDecorator` supaya bisa dibandingkan.

In [ ]:
PDF_FOLDER = "data/for_benchmarking"  # folder PDF yang diuji
pdfs = find_pdfs(PDF_FOLDER)
print(len(pdfs), "PDF")

per_halaman = TimerDecorator(chunk_per_page, verbose=False)
rekursif = TimerDecorator(
    lambda p: chunk_recursive(p, chunk_size=4000, overlap=450), verbose=False
)

chunks_pages, chunks_recursive = [], []
for pdf in pdfs:
    chunks_pages.extend(per_halaman(pdf))
    chunks_recursive.extend(rekursif(pdf))

print("per halaman:", length_stats(chunks_pages))
print("rekursif   :", length_stats(chunks_recursive))
save_times_csv(
    {
        "recursive_chunking": rekursif.get_all_times(),
        "per_page_chunking": per_halaman.get_all_times(),
    },
    "outputs/latency_chunk_results.csv",
)

## 2. Embedding

`embed_records` menambahkan vektor float32 (bytes, format storage `hash` Redis) ke tiap record; `durations` mencatat lama tiap panggilan embedding.

In [ ]:
latency_embed_recursive = []
records_recursive = embed_records(
    chunks_to_records(chunks_recursive), ai, durations=latency_embed_recursive
)

latency_embed_pages = []
records_pages = embed_records(
    chunks_to_records(chunks_pages), ai, durations=latency_embed_pages
)

save_times_csv(
    {"recursive": latency_embed_recursive},
    "outputs/latency_embedding_recursive_results.csv",
)
save_times_csv(
    {"per_page": latency_embed_pages}, "outputs/latency_embedding_page_results.csv"
)

## 3. Index di Redis

`create_index(overwrite=False)` membiarkan index lama; `insert_data` memasukkan per batch dan mengembalikan kunci yang tersimpan.

In [ ]:
db = RedisSettings.from_env().connect()
print(db.check_requirements())

schema_recursive = redis_schema("recursives_redis_sgd", prefix="recursive", dims=DIMS)
schema_pages = redis_schema("pages_redis_sgd", prefix="page", dims=DIMS)

insert_timer = TimerDecorator(db.insert_data, verbose=False)
db.create_index(schema_recursive, overwrite=False)
insert_timer(records_recursive, batch_size=100)
db.create_index(schema_pages, overwrite=False)
insert_timer(records_pages, batch_size=100)
print("lama insert (rekursif, halaman):", insert_timer.get_all_times())

## 4. Latensi query per panjang pertanyaan

Daftar pertanyaan dibaca dari `data/latency_questions.json` (contohnya ada di `data/latency_questions.example.json`). Tiap kelompok diulang beberapa kali; `workers` > 1 mengukur throughput paralel (QPS).

In [ ]:
with open("data/latency_questions.json", encoding="utf-8") as f:
    pertanyaan = json.load(f)  # {"short": [...], "medium": [...], ...}

hasil = []
for nama_index, schema in [("recursive", schema_recursive), ("pages", schema_pages)]:
    db.create_index(schema, overwrite=False)
    retriever = RedisRetriever(db, ai, mode="rrf", rrf_k=100)
    for kelompok, daftar in pertanyaan.items():
        stats = measure_latency(
            lambda q, r=retriever: r.retrieve(q, top_k=3),
            daftar,
            repeats=100,
            workers=1,
            label=f"{nama_index}-{kelompok}",
        )
        hasil.append(stats.as_dict())
        print(stats)

pd.DataFrame(hasil).to_csv("outputs/latency_query_results_redis.csv", index=False)
pd.DataFrame(hasil)